# Class 6 — FDE Data Validation with FlashEats

Yesterday, you proved you could **retrieve** data from multiple systems.

Today the question is different:

> **Can we safely use that data to make a business decision?**

This is not a generic data-cleaning lab.

## FDE validation mindset

**Business claim → assumptions → validation contract → targeted checks → stakeholder clarification → PASS / WARN / FAIL → publish decision**

### Concepts

**1. Validation is decision-dependent**  
A field can be good enough for weekly reporting but unsafe for a live operational decision.

**2. Turn assumptions into contracts**  
Business assumption → data expectation → executable check → action.

**3. Separate three validation types**
- Technical: chronology, IDs, categories, mappings
- Semantic: what does a status or ETA actually mean?
- Organizational: who owns the KPI definition?

**4. Never silently fix ambiguity**  
Do not encode thresholds or category mappings without ownership.

**5. Output a validation gate**  
The deliverable is not merely a cleaned dataframe. It is a decision:
PASS / WARN / FAIL / UNKNOWN.

In [20]:
from pathlib import Path
import zipfile

zip_path = Path("/content/flasheats-classroom-pack.zip")
extract_dir = Path("/content/flasheats")

print("ZIP exists:", zip_path.exists())

# Extract the correct ZIP
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

print("Extraction complete.")

# Find the database
db_files = list(extract_dir.rglob("flasheats.db"))

print("Database files found:")
for db in db_files:
    print(db)

if not db_files:
    raise FileNotFoundError("flasheats.db was not found after extraction.")

DB_PATH = db_files[0]
BASE = DB_PATH.parent.parent

print("\nBASE:", BASE)
print("DB_PATH:", DB_PATH)
print("Database exists:", DB_PATH.exists())

ZIP exists: True
Extraction complete.
Database files found:
/content/flasheats/flasheats-classroom-pack/database/flasheats.db
/content/flasheats/FlashEats_Classroom_Pack_V2/database/flasheats.db

BASE: /content/flasheats/flasheats-classroom-pack
DB_PATH: /content/flasheats/flasheats-classroom-pack/database/flasheats.db
Database exists: True


## Load the updated FlashEats data

In [24]:
import sqlite3
import pandas as pd
import json

# Correct dataset
BASE = Path("/content/flasheats/flasheats-classroom-pack")
DB_PATH = BASE / "database" / "flasheats.db"

print("BASE:", BASE)
print("DB_PATH:", DB_PATH)
print("Database exists:", DB_PATH.exists())

# Connect
con = sqlite3.connect(str(DB_PATH))

# Load database tables
orders = pd.read_sql("SELECT * FROM orders", con)
restaurants = pd.read_sql("SELECT * FROM restaurants", con)
drivers = pd.read_sql("SELECT * FROM drivers", con)

# Load CSV files
tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")

# Load metric definitions
with open(BASE / "data" / "client_metric_definitions.json", "r") as f:
    metric_notes = json.load(f)

print("\nLoaded successfully!")
print("orders:", orders.shape)
print("restaurants:", restaurants.shape)
print("drivers:", drivers.shape)
print("tickets:", tickets.shape)
print("restaurant_status:", restaurant_status.shape)

display(orders.head())
display(tickets.head())
display(restaurant_status.head())

BASE: /content/flasheats/flasheats-classroom-pack
DB_PATH: /content/flasheats/flasheats-classroom-pack/database/flasheats.db
Database exists: True

Loaded successfully!
orders: (1603, 13)
restaurants: (60, 6)
drivers: (120, 4)
tickets: (202, 5)
restaurant_status: (502, 4)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear
2,O00003,C0855,R010,D039,Bengaluru,2026-08-01T19:35:00,2026-08-01T20:17:52.941967,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191,delivered,2.42,medium,clear
3,O00004,C0229,R030,D038,Bengaluru,2026-08-09T18:16:00,2026-08-09T19:30:19.067006,2026-08-09T18:46:48.725485,None,cancelled,14.20,high,rain
4,O00005,C0040,R004,D047,Bengaluru,2026-08-06T20:35:00,2026-08-06T21:54:48,2026-08-06T21:18:38.024280,2026-08-06T22:15:49.290713,delivered,18.00,high,clear


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.
2,T00003,NaN,2026-08-25T18:22:00,status_mismatch,The app says picking up but the restaurant says it is ready.
3,T00004,O00037,2026-08-09T21:09:00,Late Delivery,The rider has not moved for 15 minutes.
4,T00005,O00040,2026-08-11T20:15:00,late_delivery,My order is already past the promised time.


,order_id,restaurant_id,status,last_updated_at
0,O00676,R040,READY,2026-08-17T13:29:00
1,O01506,R055,Ready,2026-08-27T16:37:00
2,O01484,R032,ready,2026-08-08T19:44:00
3,O00675,R012,handoff,2026-08-22T21:42:00
4,O00078,R027,unknown,2026-08-25T19:35:00


# Challenge 1 — Can we defend the “56% late” claim?


Leadership says:

> **“Late Delivery Rate is 56%.”**

Before calculating anything, create a validation contract.

| Business assumption | Data expectation | How will you test it? | Severity if false |
|---|---|---|---|
| One row = one business order |  |  |  |
| Delivered orders have completion time |  |  |  |
| Promised ETA is valid |  |  |  |
| Event chronology is valid |  |  |  |
| “Late” has an agreed definition |  |  |  |

**Hint:** Don't start by cleaning. Ask: *what could make 56% misleading?*

In [25]:
print("Rows:", len(orders))
print("Unique orders:", orders["order_id"].nunique())
print(orders["final_status"].value_counts(dropna=False))

# TODO:
# - establish business grain
# - check delivered orders with missing completion time
# - test promised_eta >= created_at
# - test pickup_at <= actual_delivery_at
# - mark which assumptions need stakeholder clarification

Rows: 1603
Unique orders: 1600
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64


### Challenge 1 — Can we defend the “56% late” claim?

Before calculating Late Delivery Rate, I established a validation contract to identify conditions that could make the 56% claim misleading.

| Business assumption                   | Data expectation                                                                        | How will you test it?                                                                | Severity if false |
| ------------------------------------- | --------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------ | ----------------- |
| One row = one business order          | Each `order_id` should appear exactly once                                              | Compare total rows with unique `order_id` count and identify duplicate IDs           | **Critical**      |
| Delivered orders have completion time | Every delivered order should have a valid `actual_delivery_at`                          | Filter `final_status = delivered` and count missing/unparseable `actual_delivery_at` | **Critical**      |
| Promised ETA is valid                 | `promised_eta` should be present, parseable, and not earlier than `created_at`          | Check null/invalid timestamps and test `promised_eta >= created_at`                  | **Critical**      |
| Event chronology is valid             | Events should occur in a logical order: `created_at <= pickup_at <= actual_delivery_at` | Parse timestamps and count chronology violations for applicable orders               | **High**          |
| “Late” has an agreed definition       | The business must agree on the lateness threshold and denominator                       | Compare stakeholder definitions and identify unresolved differences                  | **Critical**      |

The goal is not to immediately clean or remove problematic records. First, I will quantify the issues because duplicates, missing timestamps, invalid chronology, or an ambiguous KPI definition could materially change the reported 56% Late Delivery Rate.


In [26]:
import pandas as pd

# Make a copy so we don't modify the raw dataset
validation = orders.copy()

# Parse timestamps
time_cols = [
    "created_at",
    "promised_eta",
    "pickup_at",
    "actual_delivery_at"
]

for col in time_cols:
    validation[col] = pd.to_datetime(
        validation[col],
        errors="coerce"
    )

print("=== CHALLENGE 1 VALIDATION ===\n")

# --------------------------------------------------
# 1. Business grain: one row = one business order
# --------------------------------------------------

duplicate_rows = validation["order_id"].duplicated(keep=False)
duplicate_orders = validation.loc[
    duplicate_rows, "order_id"
].unique()

print("1. BUSINESS GRAIN")
print("Total rows:", len(validation))
print("Unique orders:", validation["order_id"].nunique())
print("Duplicate order IDs:", len(duplicate_orders))
print("Duplicate rows:", duplicate_rows.sum())

if len(duplicate_orders) > 0:
    print("Duplicate IDs:", list(duplicate_orders))

print()


# --------------------------------------------------
# 2. Delivered orders must have completion time
# --------------------------------------------------

delivered = validation[
    validation["final_status"].eq("delivered")
]

missing_completion = delivered["actual_delivery_at"].isna()

print("2. DELIVERED ORDERS → COMPLETION TIME")
print("Delivered orders:", len(delivered))
print("Missing actual_delivery_at:", missing_completion.sum())
print(
    "Completion coverage:",
    f"{(1 - missing_completion.mean()) * 100:.2f}%"
)

print()


# --------------------------------------------------
# 3. Promised ETA must be valid
# --------------------------------------------------

missing_eta = validation["promised_eta"].isna()

eta_before_creation = (
    validation["promised_eta"] < validation["created_at"]
)

print("3. PROMISED ETA VALIDITY")
print("Missing promised_eta:", missing_eta.sum())
print(
    "promised_eta before created_at:",
    eta_before_creation.sum()
)

print()


# --------------------------------------------------
# 4. Event chronology
# --------------------------------------------------

# Only test rows where both timestamps exist
pickup_before_delivery = (
    validation["pickup_at"].notna()
    & validation["actual_delivery_at"].notna()
    & (
        validation["pickup_at"]
        > validation["actual_delivery_at"]
    )
)

created_after_pickup = (
    validation["created_at"].notna()
    & validation["pickup_at"].notna()
    & (
        validation["created_at"]
        > validation["pickup_at"]
    )
)

created_after_delivery = (
    validation["created_at"].notna()
    & validation["actual_delivery_at"].notna()
    & (
        validation["created_at"]
        > validation["actual_delivery_at"]
    )
)

print("4. EVENT CHRONOLOGY")
print(
    "pickup_at > actual_delivery_at:",
    pickup_before_delivery.sum()
)
print(
    "created_at > pickup_at:",
    created_after_pickup.sum()
)
print(
    "created_at > actual_delivery_at:",
    created_after_delivery.sum()
)

print()


# --------------------------------------------------
# 5. Status distribution
# --------------------------------------------------

print("5. ORDER STATUS")
print(validation["final_status"].value_counts(dropna=False))

=== CHALLENGE 1 VALIDATION ===

1. BUSINESS GRAIN
Total rows: 1603
Unique orders: 1600
Duplicate order IDs: 3
Duplicate rows: 6
Duplicate IDs: ['O00120', 'O00723', 'O01302']

2. DELIVERED ORDERS → COMPLETION TIME
Delivered orders: 1530
Missing actual_delivery_at: 37
Completion coverage: 97.58%

3. PROMISED ETA VALIDITY
Missing promised_eta: 790
promised_eta before created_at: 4

4. EVENT CHRONOLOGY
pickup_at > actual_delivery_at: 5
created_at > pickup_at: 0
created_at > actual_delivery_at: 0

5. ORDER STATUS
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64


In [27]:
duplicates = orders[
    orders["order_id"].duplicated(keep=False)
].sort_values("order_id")

display(duplicates)

,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
119,O00120,C0501,R040,D045,Bengaluru,2026-08-13T18:15:00,2026-08-13T19:40:24,2026-08-13T18:29:08.055622,2026-08-13T19:34:08.431508,delivered,18.00,severe,clear
1600,O00120,C0501,R040,D045,Bengaluru,2026-08-13T18:15:00,2026-08-13T19:40:24,2026-08-13T18:29:08.055622,2026-08-13T19:34:08.431508,delivered,18.00,medium,clear
722,O00723,C0467,R028,D068,Bengaluru,2026-08-05T21:04:00,2026-08-05T22:19:36,2026-08-05T21:15:24.433874,2026-08-05T22:04:30.928322,delivered,18.00,medium,clear
1601,O00723,C0467,R028,D068,Bengaluru,2026-08-05T21:04:00,2026-08-05T22:19:36,2026-08-05T21:15:24.433874,2026-08-05T22:04:30.928322,delivered,18.00,high,clear
1301,O01302,C0116,R010,D088,Bengaluru,2026-08-26T21:07:00,2026-08-26T22:15:50.273095,2026-08-26T21:35:26.474087,2026-08-26T22:20:27.428561,delivered,14.78,medium,clear
1602,O01302,C0116,R010,D088,Bengaluru,2026-08-26T21:07:00,2026-08-26T22:15:50.273095,2026-08-26T21:35:26.474087,2026-08-26T22:20:27.428561,delivered,14.78,high,clear


In [28]:
chronology_issues = validation[
    validation["pickup_at"].notna()
    & validation["actual_delivery_at"].notna()
    & (validation["pickup_at"] > validation["actual_delivery_at"])
]

display(
    chronology_issues[
        [
            "order_id",
            "created_at",
            "pickup_at",
            "actual_delivery_at",
            "final_status"
        ]
    ]
)

,order_id,created_at,pickup_at,actual_delivery_at,final_status
50,O00051,2026-08-01 15:09:00,2026-08-01 16:24:51.495665,2026-08-01 16:19:51.495665,delivered
51,O00052,2026-08-24 17:07:00,2026-08-24 18:16:16.946011,2026-08-24 18:11:16.946011,delivered
52,O00053,2026-08-13 17:06:00,2026-08-13 18:25:53.797613,2026-08-13 18:20:53.797613,delivered
53,O00054,2026-08-28 19:56:00,2026-08-28 21:25:10.582862,2026-08-28 21:20:10.582862,delivered
54,O00055,2026-08-16 12:53:00,2026-08-16 14:08:05.645770,2026-08-16 14:03:05.645770,delivered


In [29]:
eta_issues = validation[
    validation["promised_eta"].notna()
    & validation["created_at"].notna()
    & (validation["promised_eta"] < validation["created_at"])
]

display(
    eta_issues[
        [
            "order_id",
            "created_at",
            "promised_eta",
            "pickup_at",
            "actual_delivery_at",
            "final_status"
        ]
    ]
)

,order_id,created_at,promised_eta,pickup_at,actual_delivery_at,final_status
100,O00101,2026-08-24 20:59:00,2026-08-24 20:49:00,2026-08-24 21:25:03.983984,2026-08-24 22:14:30.685278,delivered
101,O00102,2026-08-11 16:41:00,2026-08-11 16:31:00,2026-08-11 17:05:05.809584,2026-08-11 17:57:17.661772,delivered
102,O00103,2026-08-22 20:11:00,2026-08-22 20:01:00,2026-08-22 20:39:45.064627,2026-08-22 21:10:17.342305,delivered
103,O00104,2026-08-09 22:19:00,2026-08-09 22:09:00,2026-08-09 22:47:33.934776,2026-08-09 23:36:20.740161,delivered


### Validation Findings

The initial validation shows that the raw dataset does not fully satisfy the assumptions required to defend the reported 56% Late Delivery Rate.

* **Business grain:** There are 1,603 rows but only 1,600 unique `order_id` values. Three order IDs (`O00120`, `O00723`, and `O01302`) appear twice, indicating duplicate representations of business orders.
* **Completion timestamp:** Of 1,530 orders marked `delivered`, 37 have a missing `actual_delivery_at`. These orders cannot be reliably classified as late or on-time using delivery completion time.
* **Promised ETA:** 790 records have a missing `promised_eta`, and 4 records have a `promised_eta` earlier than `created_at`. Therefore, ETA-based lateness cannot be evaluated for a large portion of the dataset without making assumptions.
* **Event chronology:** Five records have `pickup_at` later than `actual_delivery_at`, violating the expected order of events. These records require investigation rather than silent correction.
* **Status consistency:** The dataset contains both `delivered` and `Delivered`, indicating a potential status-standardization issue that could affect filtering.
* **Late definition:** The business definition is not fully agreed. Operations considers any delivery after the promised ETA late, while Support considers more than 10 minutes beyond ETA meaningfully late. The denominator also needs to be explicitly defined, particularly regarding cancelled orders and delivered orders with missing timestamps.

### Conclusion

The “56% late” claim cannot be considered fully defensible from the raw data alone.

The main reasons are the duplicate order records, missing completion timestamps, a large number of missing promised ETAs, chronology violations, inconsistent status values, and the absence of a single agreed definition and denominator for Late Delivery Rate.

Importantly, these records should not be silently removed or corrected before their impact is quantified. The next step is to establish the canonical KPI definition and measure how many orders are actually eligible for the calculation and how the reported rate changes under the agreed definition.


In [30]:
# Normalize status ONLY for inspection
status_normalized = orders["final_status"].astype(str).str.strip().str.lower()

delivered_all = status_normalized.eq("delivered")

print("=== KPI ELIGIBILITY CHECK ===")

print("Total rows:", len(orders))
print("Unique orders:", orders["order_id"].nunique())

print("\nDelivered after case normalization:", delivered_all.sum())

print(
    "Delivered with actual_delivery_at:",
    orders.loc[delivered_all, "actual_delivery_at"].notna().sum()
)

print(
    "Delivered with promised_eta:",
    orders.loc[delivered_all, "promised_eta"].notna().sum()
)

print(
    "Delivered with BOTH timestamps:",
    (
        orders.loc[delivered_all, "actual_delivery_at"].notna()
        &
        orders.loc[delivered_all, "promised_eta"].notna()
    ).sum()
)

=== KPI ELIGIBILITY CHECK ===
Total rows: 1603
Unique orders: 1600

Delivered after case normalization: 1535
Delivered with actual_delivery_at: 1498
Delivered with promised_eta: 1535
Delivered with BOTH timestamps: 1498


In [31]:
# Normalize status for KPI calculation
status = orders["final_status"].astype(str).str.strip().str.lower()

# Parse timestamps
created = pd.to_datetime(orders["created_at"], errors="coerce")
eta = pd.to_datetime(orders["promised_eta"], errors="coerce")
actual = pd.to_datetime(orders["actual_delivery_at"], errors="coerce")

# Eligible population:
# delivered + both ETA and actual delivery timestamps
eligible = (
    status.eq("delivered")
    & eta.notna()
    & actual.notna()
)

kpi_data = orders.loc[eligible].copy()

kpi_data["promised_eta"] = eta.loc[eligible]
kpi_data["actual_delivery_at"] = actual.loc[eligible]

# Delay in minutes
kpi_data["delay_min"] = (
    kpi_data["actual_delivery_at"]
    - kpi_data["promised_eta"]
).dt.total_seconds() / 60

# Definition: any delivery after promised ETA = late
kpi_data["late_flag"] = kpi_data["delay_min"] > 0

late_count = kpi_data["late_flag"].sum()
eligible_count = len(kpi_data)
late_rate = late_count / eligible_count * 100

print("=== LATE DELIVERY RATE ===")
print("Eligible delivered orders:", eligible_count)
print("Late orders:", late_count)
print("On-time orders:", eligible_count - late_count)
print(f"Late Delivery Rate: {late_rate:.2f}%")

=== LATE DELIVERY RATE ===
Eligible delivered orders: 756
Late orders: 430
On-time orders: 326
Late Delivery Rate: 56.88%


In [32]:
# Support definition:
# more than 10 minutes beyond promised ETA = late

kpi_data["late_10min_flag"] = kpi_data["delay_min"] > 10

late_10_count = kpi_data["late_10min_flag"].sum()
late_10_rate = late_10_count / eligible_count * 100

print("=== ALTERNATIVE DEFINITION ===")
print("Threshold: > 10 minutes late")
print("Late orders:", late_10_count)
print(f"Late Delivery Rate: {late_10_rate:.2f}%")

=== ALTERNATIVE DEFINITION ===
Threshold: > 10 minutes late
Late orders: 183
Late Delivery Rate: 24.21%


### KPI Calculation and Sensitivity

Using the Operations definition that considers an order late when `actual_delivery_at > promised_eta`, the calculation produces:

* Eligible delivered orders: **756**
* Late orders: **430**
* On-time orders: **326**
* Late Delivery Rate: **56.88%**

This is close to leadership's reported **56%** figure.

However, the result should be interpreted as the Late Delivery Rate for the **eligible population used in the calculation**, rather than automatically treating it as the definitive company-wide KPI.

The dataset contains **1,535 delivered records after normalizing the status values**, but only 756 were included in the final calculation. The excluded records therefore need to be understood before the KPI can be considered fully reliable.

### Definition Sensitivity

The result changes substantially when applying the Support definition of meaningful lateness:

* Threshold: **more than 10 minutes beyond promised ETA**
* Eligible delivered orders: **756**
* Late orders: **183**
* Late Delivery Rate: **24.21%**

Therefore, the reported percentage is highly sensitive to the agreed definition of “late”.

### Final Conclusion

The 56% claim is **numerically reproducible under the Operations definition on the eligible population**, with a calculated rate of **56.88%**.

However, the claim is not fully defensible as a canonical KPI until the business confirms:

1. The official definition of “late” — any delay versus more than 10 minutes.
2. The denominator — which delivered orders are eligible for the KPI.
3. How delivered orders with missing `actual_delivery_at` should be treated.
4. How duplicate order records should be handled.
5. How inconsistent status values such as `delivered` and `Delivered` should be standardized.

The key FDE finding is that **the 56% figure is not simply a property of the raw data; it depends on business rules, data eligibility, and the definition of the metric.**


In [33]:
print("Total delivered:", delivered_all.sum())

print("\nDelivered missing actual_delivery_at:",
      (
          delivered_all
          & pd.to_datetime(orders["actual_delivery_at"], errors="coerce").isna()
      ).sum())

print("\nDelivered missing promised_eta:",
      (
          delivered_all
          & pd.to_datetime(orders["promised_eta"], errors="coerce").isna()
      ).sum())

print("\nDelivered with BOTH timestamps:",
      (
          delivered_all
          & pd.to_datetime(orders["actual_delivery_at"], errors="coerce").notna()
          & pd.to_datetime(orders["promised_eta"], errors="coerce").notna()
      ).sum())

Total delivered: 1535

Delivered missing actual_delivery_at: 37

Delivered missing promised_eta: 760

Delivered with BOTH timestamps: 756


### Final Conclusion

The reported **56% Late Delivery Rate is numerically reproducible under the Operations definition**, where an order is considered late when `actual_delivery_at > promised_eta`.

For the **756 delivered orders with both a valid promised ETA and actual delivery timestamp**, 430 were late, producing a Late Delivery Rate of **56.88%**. This is close to the leadership claim of 56%.

However, the claim cannot be treated as a fully validated KPI without resolving the data and business-definition issues identified during validation.

There are **1,535 delivered orders**, but only **756 have both timestamps required to determine lateness**. Therefore, 779 delivered records cannot currently be classified using the same rule. The dataset also contains 3 duplicate order IDs, 37 delivered orders missing completion timestamps, 4 records where `promised_eta` occurs before `created_at`, 5 chronology violations where pickup occurs after delivery, and inconsistent status capitalization (`delivered` versus `Delivered`).

The definition of “late” also materially changes the result. Using the Operations definition gives **56.88%**, while using the Support definition of more than 10 minutes beyond ETA gives **24.21%**.

Therefore, the key finding is:

> **The 56% figure can be reproduced for the eligible population under one business definition, but it should not be presented as a definitive KPI until the business agrees on the lateness threshold, denominator, treatment of missing timestamps, duplicate records, and status standardization.**

This validation demonstrates why the metric must be defined and its eligible population established before using the percentage for operational decisions.


# Challenge 2 — Stakeholders disagree on “late”


Read `client_metric_definitions.json`.

Calculate the metric under at least three definitions:
- any delay > 0 minutes,
- delay > 10 minutes,
- historical-style delivered/non-null population.

| Definition | Late rate | Business meaning |
|---|---:|---|

Then answer:

> **Which one should leadership publish, and who must own that decision?**

In [34]:
print(json.dumps(metric_notes, indent=2))

{
  "metric_under_review": "Late Delivery Rate",
  "leadership_claim": "Late Delivery Rate is 56%",
  "stakeholders": {
    "VP Operations": "Any delivered order after the promised ETA is late.",
    "Support Lead": "Only more than 10 minutes beyond ETA should count as meaningfully late.",
    "Finance": "Cancelled/refunded orders should not count in operational performance.",
    "Data Team": "Historical dashboard uses delivered orders with non-null actual delivery time."
  },
  "note": "No canonical KPI owner is formally documented."
}


In [35]:
analysis = orders.drop_duplicates("order_id", keep="first").copy()

# Normalize status
analysis["status_normalized"] = (
    analysis["final_status"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Parse timestamps
for c in ["promised_eta", "actual_delivery_at"]:
    analysis[c] = pd.to_datetime(
        analysis[c],
        format="mixed",
        errors="coerce"
    )

# Calculate delay
analysis["delay_min"] = (
    analysis["actual_delivery_at"] -
    analysis["promised_eta"]
).dt.total_seconds() / 60


# ============================================================
# DEFINITION 1: Any delay > 0 minutes
# ============================================================

definition_1 = analysis[
    (analysis["status_normalized"] == "delivered") &
    analysis["promised_eta"].notna() &
    analysis["actual_delivery_at"].notna()
].copy()

late_1 = definition_1["delay_min"] > 0

rate_1 = late_1.mean() * 100


# ============================================================
# DEFINITION 2: Delay > 10 minutes
# ============================================================

definition_2 = definition_1.copy()

late_2 = definition_2["delay_min"] > 10

rate_2 = late_2.mean() * 100


# ============================================================
# DEFINITION 3: Historical-style population
# Delivered + non-null actual delivery time
# ============================================================

definition_3 = analysis[
    (analysis["status_normalized"] == "delivered") &
    analysis["actual_delivery_at"].notna()
].copy()

late_3 = (
    definition_3["actual_delivery_at"] >
    definition_3["promised_eta"]
)

# Only rows where ETA exists can actually be classified as late/on-time
rate_3 = late_3.mean() * 100


# ============================================================
# PRINT RESULTS
# ============================================================

print("=== CHALLENGE 2: DEFINITION SENSITIVITY ===\n")

print("Definition 1: Any delay > 0 minutes")
print("Eligible orders:", len(definition_1))
print("Late orders:", late_1.sum())
print(f"Late rate: {rate_1:.2f}%\n")

print("Definition 2: Delay > 10 minutes")
print("Eligible orders:", len(definition_2))
print("Late orders:", late_2.sum())
print(f"Late rate: {rate_2:.2f}%\n")

print("Definition 3: Historical-style delivered/non-null population")
print("Delivered orders:", len(definition_3))
print("Late orders:", late_3.sum())
print(f"Late rate: {rate_3:.2f}%")

=== CHALLENGE 2: DEFINITION SENSITIVITY ===

Definition 1: Any delay > 0 minutes
Eligible orders: 1495
Late orders: 843
Late rate: 56.39%

Definition 2: Delay > 10 minutes
Eligible orders: 1495
Late orders: 349
Late rate: 23.34%

Definition 3: Historical-style delivered/non-null population
Delivered orders: 1495
Late orders: 843
Late rate: 56.39%


In [36]:
historical_eligible = definition_3[
    definition_3["promised_eta"].notna()
].copy()

historical_late = (
    historical_eligible["actual_delivery_at"] >
    historical_eligible["promised_eta"]
)

historical_rate = historical_late.mean() * 100

print("Historical-style eligible:", len(historical_eligible))
print("Historical-style late:", historical_late.sum())
print(f"Historical-style late rate: {historical_rate:.2f}%")

Historical-style eligible: 1495
Historical-style late: 843
Historical-style late rate: 56.39%


### Challenge 2 — Stakeholder Definition Analysis

The stakeholder definitions produce materially different Late Delivery Rates:

| Definition                                     |  Late rate | Business meaning                                                                                                            |
| ---------------------------------------------- | ---------: | --------------------------------------------------------------------------------------------------------------------------- |
| Any delay > 0 minutes                          | **56.39%** | Any delivered order after its promised ETA is considered late. This matches the VP Operations definition.                   |
| Delay > 10 minutes                             | **23.34%** | Only deliveries more than 10 minutes beyond ETA are considered meaningfully late. This matches the Support Lead definition. |
| Historical-style delivered/non-null population | **56.39%** | Uses the historical-style delivered population with non-null actual delivery time.                                          |

The difference between 56.39% and 23.34% shows that the reported KPI is highly dependent on the business definition of “late.”

### KPI ownership

Leadership should publish **one clearly documented canonical Late Delivery Rate**, but the choice of threshold and denominator should be made by the business rather than inferred by the Data Team.

The metric should have a formally assigned **KPI owner**, with Operations, Support, Finance, and Data agreeing on:

* what “late” means,
* which orders belong in the denominator,
* how cancelled/refunded orders are treated,
* how missing timestamps are handled, and
* how the definition is maintained over time.

The current metric definitions explicitly state that **no canonical KPI owner is formally documented**, so this is an unresolved governance issue.

Until the definition is formally agreed, reporting should label the metric with its definition and population rather than presenting “Late Delivery Rate is 56%” without qualification.

### FDE takeaway

The analysis shows that the disagreement is not merely a data-calculation problem. Changing the lateness threshold changes the reported rate from **56.39% to 23.34%**. Therefore, a trustworthy KPI requires both a reliable data pipeline and an explicitly owned business definition.


# Challenge 3 — Validate categories without cleaning by instinct


Inspect:
- `final_status`
- `traffic_bucket`
- restaurant `status`
- support-ticket `category`

For each:
1. list observed values,
2. identify representation differences,
3. decide what is safe to normalize,
4. identify what requires owner confirmation.

Remember:

> `"Delivered"` vs `"delivered"` is likely representation.  
> `"handoff"` vs `"handed_off"` may be semantics.

In [38]:
for col in ["final_status", "traffic_bucket"]:
    print("\n", col)
    print(orders[col].value_counts(dropna=False))

print("\nRestaurant status")
print(restaurant_status["status"].value_counts(dropna=False))

print("\nSupport category")
print(tickets["category"].value_counts(dropna=False))


 final_status
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64

 traffic_bucket
traffic_bucket
medium    635
high      448
low       381
severe    136
HIGH        3
Name: count, dtype: int64

Restaurant status
status
preparing     171
handed_off    168
ready         157
ready           2
handoff         1
Ready           1
READY           1
unknown         1
Name: count, dtype: int64

Support category
category
late_delivery        38
eta_changed          37
restaurant_delay     37
ready_but_waiting    33
status_mismatch      29
driver_not_moving    25
Late Delivery         1
late_delivery         1
ETA issue             1
Name: count, dtype: int64


In [39]:
orders["final_status_normalized"] = (
    orders["final_status"].astype(str).str.strip().str.lower()
)

In [40]:
traffic_bucket_normalized = (
    orders["traffic_bucket"].astype(str).str.strip().str.lower()
)

In [41]:
print("Restaurant status — raw representations:")
for value in restaurant_status["status"].dropna().unique():
    print(repr(value))

Restaurant status — raw representations:
'READY'
'Ready'
'ready '
'handoff'
'unknown'
'handed_off'
'ready'
'preparing'


In [42]:
print("Support categories — raw representations:")
for value in tickets["category"].dropna().unique():
    print(repr(value))

Support categories — raw representations:
'late_delivery'
'eta_changed'
'status_mismatch'
'Late Delivery'
'late_delivery '
'ETA issue'
'restaurant_delay'
'ready_but_waiting'
'driver_not_moving'


### Challenge 3 — Category Validation

I inspected the raw categorical values before applying any normalization. The goal was to distinguish representation differences from differences that may have business meaning.

| Field                     | Observed values                                                                                                                                                 | Safe normalization                                                | Owner confirmation required                                                                                                     |
| ------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------- |
| `final_status`            | `delivered`, `Delivered`, `cancelled`                                                                                                                           | Normalize whitespace and casing: `Delivered` → `delivered`        | Confirm that the casing difference has no semantic meaning                                                                      |
| `traffic_bucket`          | `low`, `medium`, `high`, `HIGH`, `severe`                                                                                                                       | Normalize whitespace and casing: `HIGH` → `high`                  | Do not merge `high` and `severe`; confirm the business meaning of traffic thresholds if needed                                  |
| Restaurant `status`       | `READY`, `Ready`, `ready `, `ready`, `handoff`, `handed_off`, `preparing`, `unknown`                                                                            | Normalize whitespace/casing: `READY`, `Ready`, `ready ` → `ready` | `handoff` vs `handed_off` may represent different business events and must not be merged without confirmation; retain `unknown` |
| Support-ticket `category` | `late_delivery`, `Late Delivery`, `late_delivery `, `eta_changed`, `ETA issue`, `status_mismatch`, `restaurant_delay`, `ready_but_waiting`, `driver_not_moving` | Normalize `Late Delivery` and `late_delivery ` → `late_delivery`  | `eta_changed` vs `ETA issue` may have different meanings and require owner confirmation                                         |

### Representation differences

The raw values confirm several formatting inconsistencies:

* `Delivered` vs `delivered`
* `HIGH` vs `high`
* `READY`, `Ready`, `ready ` vs `ready`
* `Late Delivery`, `late_delivery ` vs `late_delivery`

These can safely be standardized using whitespace trimming and consistent casing when the business meaning is unchanged.

### Semantic differences

Some categories should **not** be merged automatically:

* `handoff` vs `handed_off`
* `eta_changed` vs `ETA issue`
* `high` vs `severe`

The names alone do not establish that these categories represent the same business event or state. They should remain distinct until the relevant business/data owner confirms their intended meaning.

The `unknown` restaurant status should also be preserved rather than reassigned to another category without evidence.

### FDE takeaway

Category normalization should remove **representation noise**, not change **business semantics**. Formatting inconsistencies can be normalized from the data itself, while semantic mappings require explicit business-owner confirmation.


# Challenge 4 — Cross-source integrity


Validate mappings:
- `orders.restaurant_id` → restaurants
- `orders.driver_id` → drivers
- `tickets.order_id` → orders
- `restaurant_status.order_id` → orders

Produce:

| Relationship | Coverage % | Status | Risk |
|---|---:|---|---|

Then discuss:

> If 1% is unmapped, is that acceptable?

It depends on which business decision uses those records.

In [43]:
# Challenge 4 — Cross-source integrity

print("=== CHALLENGE 4: CROSS-SOURCE INTEGRITY ===\n")


# ------------------------------------------------------------
# 1. orders.restaurant_id → restaurants
# ------------------------------------------------------------

restaurant_ids = set(restaurants["restaurant_id"].dropna().astype(str))

order_restaurant_ids = orders["restaurant_id"].dropna().astype(str)

restaurant_match = order_restaurant_ids.isin(restaurant_ids)

restaurant_total = len(order_restaurant_ids)
restaurant_mapped = restaurant_match.sum()
restaurant_unmapped = restaurant_total - restaurant_mapped
restaurant_coverage = restaurant_mapped / restaurant_total * 100

print("1. orders.restaurant_id → restaurants")
print("Source records:", restaurant_total)
print("Mapped:", restaurant_mapped)
print("Unmapped:", restaurant_unmapped)
print(f"Coverage: {restaurant_coverage:.2f}%")

if restaurant_unmapped > 0:
    print("Unmapped restaurant IDs:",
          sorted(order_restaurant_ids[~restaurant_match].unique()))


# ------------------------------------------------------------
# 2. orders.driver_id → drivers
# ------------------------------------------------------------

driver_ids = set(drivers["driver_id"].dropna().astype(str))

order_driver_ids = orders["driver_id"].dropna().astype(str)

driver_match = order_driver_ids.isin(driver_ids)

driver_total = len(order_driver_ids)
driver_mapped = driver_match.sum()
driver_unmapped = driver_total - driver_mapped
driver_coverage = driver_mapped / driver_total * 100

print("\n2. orders.driver_id → drivers")
print("Source records:", driver_total)
print("Mapped:", driver_mapped)
print("Unmapped:", driver_unmapped)
print(f"Coverage: {driver_coverage:.2f}%")

if driver_unmapped > 0:
    print("Unmapped driver IDs:",
          sorted(order_driver_ids[~driver_match].unique()))


# ------------------------------------------------------------
# 3. tickets.order_id → orders
# ------------------------------------------------------------

order_ids = set(orders["order_id"].dropna().astype(str))

ticket_order_ids = tickets["order_id"].dropna().astype(str)

ticket_match = ticket_order_ids.isin(order_ids)

ticket_total = len(ticket_order_ids)
ticket_mapped = ticket_match.sum()
ticket_unmapped = ticket_total - ticket_mapped
ticket_coverage = ticket_mapped / ticket_total * 100

print("\n3. tickets.order_id → orders")
print("Source records:", ticket_total)
print("Mapped:", ticket_mapped)
print("Unmapped:", ticket_unmapped)
print(f"Coverage: {ticket_coverage:.2f}%")

if ticket_unmapped > 0:
    print("Unmapped order IDs:",
          sorted(ticket_order_ids[~ticket_match].unique()))


# ------------------------------------------------------------
# 4. restaurant_status.order_id → orders
# ------------------------------------------------------------

status_order_ids = restaurant_status["order_id"].dropna().astype(str)

status_match = status_order_ids.isin(order_ids)

status_total = len(status_order_ids)
status_mapped = status_match.sum()
status_unmapped = status_total - status_mapped
status_coverage = status_mapped / status_total * 100

print("\n4. restaurant_status.order_id → orders")
print("Source records:", status_total)
print("Mapped:", status_mapped)
print("Unmapped:", status_unmapped)
print(f"Coverage: {status_coverage:.2f}%")

if status_unmapped > 0:
    print("Unmapped order IDs:",
          sorted(status_order_ids[~status_match].unique()))

=== CHALLENGE 4: CROSS-SOURCE INTEGRITY ===

1. orders.restaurant_id → restaurants
Source records: 1600
Mapped: 1600
Unmapped: 0
Coverage: 100.00%

2. orders.driver_id → drivers
Source records: 1600
Mapped: 1600
Unmapped: 0
Coverage: 100.00%

3. tickets.order_id → orders
Source records: 199
Mapped: 199
Unmapped: 0
Coverage: 100.00%

4. restaurant_status.order_id → orders
Source records: 502
Mapped: 502
Unmapped: 0
Coverage: 100.00%


### Challenge 4 — Cross-Source Integrity

All four cross-source relationships were validated by checking whether referenced IDs exist in their corresponding parent dataset.

| Relationship                          |  Coverage % | Status       | Risk                                     |
| ------------------------------------- | ----------: | ------------ | ---------------------------------------- |
| `orders.restaurant_id → restaurants`  | **100.00%** | Fully mapped | No referential-integrity risk identified |
| `orders.driver_id → drivers`          | **100.00%** | Fully mapped | No referential-integrity risk identified |
| `tickets.order_id → orders`           | **100.00%** | Fully mapped | No orphaned support-ticket references    |
| `restaurant_status.order_id → orders` | **100.00%** | Fully mapped | No orphaned restaurant-status references |

### Findings

Every non-null restaurant and driver reference in `orders` maps to an existing record in the corresponding master dataset.

Similarly, every non-null `order_id` in `tickets` and `restaurant_status` maps to an order in `orders`.

Therefore, there is **no cross-source mapping gap** for the relationships tested.

### Is 1% unmapped acceptable?

A 1% unmapped rate is not automatically acceptable or unacceptable. Its impact depends on the business decision using those records.

For example, 1% missing mappings might have little effect on a high-level KPI if the records are randomly distributed. However, if those records are concentrated in one restaurant, driver group, city, or support category, they could materially affect a segment-level decision.

Therefore, unmapped records should be evaluated based on their **volume, concentration, and business importance**, rather than using a fixed percentage threshold.

### FDE takeaway

Cross-source integrity is strong in this dataset: all tested relationships have 100% coverage. This means downstream joins between these sources should not lose records because of missing foreign-key mappings, although other data-quality issues such as duplicates, missing timestamps, and inconsistent categories still need to be handled separately.


# Challenge 5 — Freshness is an SLA question


Use `restaurant_status.csv`.

Determine whether status updates are fresh enough for:
- weekly analytics,
- live customer ETA,
- restaurant accountability.

The same record may be acceptable for one use case and unsafe for another.

**Hint:** join status records to order lifecycle timestamps and inspect timing.

In [ ]:
# TODO:
# Merge restaurant_status with orders on order_id
# Parse timestamps
# Compare last_updated_at with order lifecycle timestamps

In [44]:
# Challenge 5 — Freshness analysis

import pandas as pd

# Keep only the columns needed from orders
order_lifecycle = orders[
    [
        "order_id",
        "created_at",
        "pickup_at",
        "actual_delivery_at",
        "final_status"
    ]
].drop_duplicates("order_id", keep="first").copy()

status_analysis = restaurant_status.merge(
    order_lifecycle,
    on="order_id",
    how="left"
)

# Parse timestamps
timestamp_cols = [
    "created_at",
    "pickup_at",
    "actual_delivery_at",
    "last_updated_at"
]

for col in timestamp_cols:
    status_analysis[col] = pd.to_datetime(
        status_analysis[col],
        format="mixed",
        errors="coerce"
    )

# Calculate freshness relative to important lifecycle events
status_analysis["lag_from_created_min"] = (
    status_analysis["last_updated_at"] -
    status_analysis["created_at"]
).dt.total_seconds() / 60

status_analysis["lag_from_pickup_min"] = (
    status_analysis["last_updated_at"] -
    status_analysis["pickup_at"]
).dt.total_seconds() / 60

status_analysis["lag_from_delivery_min"] = (
    status_analysis["last_updated_at"] -
    status_analysis["actual_delivery_at"]
).dt.total_seconds() / 60


print("=== CHALLENGE 5: FRESHNESS ===\n")

print("Status records:", len(status_analysis))
print(
    "Missing last_updated_at:",
    status_analysis["last_updated_at"].isna().sum()
)

print("\nLast update relative to order creation:")
print(status_analysis["lag_from_created_min"].describe())

print("\nLast update relative to pickup:")
print(status_analysis["lag_from_pickup_min"].describe())

print("\nLast update relative to delivery:")
print(status_analysis["lag_from_delivery_min"].describe())

=== CHALLENGE 5: FRESHNESS ===

Status records: 502
Missing last_updated_at: 0

Last update relative to order creation:
count    502.000000
mean      15.922311
std       41.129559
min     -354.000000
25%       12.000000
50%       20.000000
75%       28.000000
max       35.000000
Name: lag_from_created_min, dtype: float64

Last update relative to pickup:
count    502.000000
mean     -11.225559
std       42.311393
min     -385.817233
25%      -16.790911
50%       -6.556680
75%        3.155284
max       25.386850
Name: lag_from_pickup_min, dtype: float64

Last update relative to delivery:
count    475.000000
mean     -57.621885
std       44.823288
min     -434.563241
25%      -66.276322
50%      -53.771055
75%      -40.345643
max       -5.305920
Name: lag_from_delivery_min, dtype: float64


In [45]:
print("\n=== LATEST STATUS RECORDS ===")

print(
    status_analysis[
        [
            "order_id",
            "created_at",
            "pickup_at",
            "actual_delivery_at",
            "last_updated_at",
            "status"
        ]
    ]
    .sort_values("last_updated_at")
    .tail(20)
    .to_string(index=False)
)


=== LATEST STATUS RECORDS ===
order_id          created_at                  pickup_at         actual_delivery_at     last_updated_at     status
  O01447 2026-08-28 12:57:00 2026-08-28 13:14:44.390098 2026-08-28 14:01:46.021786 2026-08-28 13:27:00      ready
  O01235 2026-08-28 13:15:00 2026-08-28 14:04:54.255957 2026-08-28 15:08:17.715971 2026-08-28 13:38:00      ready
  O00322 2026-08-28 13:18:00 2026-08-28 14:04:39.367541 2026-08-28 14:55:30.304349 2026-08-28 13:46:00  preparing
  O00678 2026-08-28 13:48:00 2026-08-28 14:21:34.657910 2026-08-28 14:55:15.036102 2026-08-28 14:07:00 handed_off
  O01114 2026-08-28 15:25:00 2026-08-28 15:47:41.871627 2026-08-28 16:41:47.961835 2026-08-28 15:57:00 handed_off
  O00901 2026-08-28 17:26:00 2026-08-28 17:55:39.325212 2026-08-28 18:43:35.316572 2026-08-28 18:01:00  preparing
  O00377 2026-08-28 17:54:00 2026-08-28 18:18:11.811990 2026-08-28 18:53:13.696174 2026-08-28 18:08:00      ready
  O01236 2026-08-28 17:51:00 2026-08-28 18:14:32.584706 2

In [46]:
print("\n=== LARGEST CREATION → STATUS UPDATE LAGS ===")

print(
    status_analysis[
        [
            "order_id",
            "created_at",
            "last_updated_at",
            "lag_from_created_min",
            "status"
        ]
    ]
    .sort_values("lag_from_created_min", ascending=False)
    .head(10)
    .to_string(index=False)
)


=== LARGEST CREATION → STATUS UPDATE LAGS ===
order_id          created_at     last_updated_at  lag_from_created_min     status
  O00901 2026-08-28 17:26:00 2026-08-28 18:01:00                  35.0  preparing
  O00490 2026-08-08 15:02:00 2026-08-08 15:37:00                  35.0      ready
  O00696 2026-08-25 22:17:00 2026-08-25 22:52:00                  35.0  preparing
  O00735 2026-08-23 18:02:00 2026-08-23 18:37:00                  35.0 handed_off
  O00332 2026-08-22 20:46:00 2026-08-22 21:21:00                  35.0  preparing
  O00518 2026-08-13 22:50:00 2026-08-13 23:25:00                  35.0      ready
  O00006 2026-08-26 23:02:00 2026-08-26 23:37:00                  35.0      ready
  O01446 2026-08-01 20:27:00 2026-08-01 21:02:00                  35.0  preparing
  O01187 2026-08-26 20:03:00 2026-08-26 20:38:00                  35.0      ready
  O00630 2026-08-06 19:34:00 2026-08-06 20:09:00                  35.0 handed_off


In [47]:
print(
    "Updates before order creation:",
    (status_analysis["lag_from_created_min"] < 0).sum()
)

Updates before order creation: 6


In [48]:
print(
    "Updates after pickup:",
    (status_analysis["lag_from_pickup_min"] > 0).sum()
)

Updates after pickup: 166


In [49]:
print("=== CHALLENGE 5: FINAL FRESHNESS SUMMARY ===")

print("Total restaurant status records:", len(status_analysis))
print("Missing last_updated_at:", status_analysis["last_updated_at"].isna().sum())

print("\nChronology checks:")
print(
    "Updates before order creation:",
    (status_analysis["lag_from_created_min"] < 0).sum()
)
print(
    "Updates after pickup:",
    (status_analysis["lag_from_pickup_min"] > 0).sum()
)

print("\nCreation → last status update:")
print("Median:", status_analysis["lag_from_created_min"].median(), "minutes")
print("75th percentile:", status_analysis["lag_from_created_min"].quantile(0.75), "minutes")
print("Maximum:", status_analysis["lag_from_created_min"].max(), "minutes")

print("\nPickup → last status update:")
print("Median:", status_analysis["lag_from_pickup_min"].median(), "minutes")
print("Maximum:", status_analysis["lag_from_pickup_min"].max(), "minutes")

print("\nDelivery → last status update:")
delivery_lag = status_analysis["lag_from_delivery_min"].dropna()
print("Records with delivery timestamp:", len(delivery_lag))
print("Median:", delivery_lag.median(), "minutes")
print("Maximum:", delivery_lag.max(), "minutes")

=== CHALLENGE 5: FINAL FRESHNESS SUMMARY ===
Total restaurant status records: 502
Missing last_updated_at: 0

Chronology checks:
Updates before order creation: 6
Updates after pickup: 166

Creation → last status update:
Median: 20.0 minutes
75th percentile: 28.0 minutes
Maximum: 35.0 minutes

Pickup → last status update:
Median: -6.556679991666667 minutes
Maximum: 25.3868499 minutes

Delivery → last status update:
Records with delivery timestamp: 475
Median: -53.771054983333336 minutes
Maximum: -5.3059199 minutes


# Challenge 5 — Freshness is an SLA Question

## Objective

Determine whether restaurant status updates are fresh enough for:

* Weekly analytics
* Live customer ETA
* Restaurant accountability

The same status record may be suitable for one use case but unsafe for another.

## Freshness Analysis

I joined `restaurant_status` with the order lifecycle timestamps and compared `last_updated_at` with order creation, pickup, and delivery times.

### Observed Results

| Metric                                        |     Result |
| --------------------------------------------- | ---------: |
| Total restaurant status records               |        502 |
| Missing `last_updated_at`                     |          0 |
| Updates before order creation                 |          6 |
| Updates after pickup                          |        166 |
| Median creation → last status update          |     20 min |
| 75th percentile creation → last status update |     28 min |
| Maximum creation → last status update         |     35 min |
| Median pickup → last status update            |  -6.56 min |
| Maximum pickup → last status update           |  25.39 min |
| Records with delivery timestamp               |        475 |
| Median delivery → last status update          | -53.77 min |
| Maximum delivery → last status update         |  -5.31 min |

## Use-Case Assessment

| Use case                      | Freshness requirement                                           | Assessment                                    | Risk                                                                                   |
| ----------------------------- | --------------------------------------------------------------- | --------------------------------------------- | -------------------------------------------------------------------------------------- |
| **Weekly analytics**          | Updates should be available before the reporting cutoff         | **Generally usable**                          | Low to moderate, provided the reporting process allows enough time for delayed updates |
| **Live customer ETA**         | Requires near-real-time and reliably ordered status information | **Not established as safe from this dataset** | Stale or delayed status could affect customer-facing ETA                               |
| **Restaurant accountability** | Status events need to be timely and correctly ordered           | **Usable with caution**                       | Delayed status updates could affect attribution of restaurant-related delays           |

## Data Quality Findings

There are **6 records where `last_updated_at` occurs before `created_at`**. This is a chronology inconsistency and should be investigated rather than silently corrected.

There are also **166 records where the restaurant status was updated after pickup**. This may indicate delayed status updates, but it should not automatically be interpreted as system ingestion delay because `last_updated_at` may represent the time the restaurant status was recorded rather than the time the underlying event actually occurred.

The fact that the median `pickup → last_updated_at` is **-6.56 minutes** also suggests that many restaurant status updates naturally occur before pickup, which is expected for statuses such as `preparing`, `ready`, and `handed_off`.

## Key Limitation

`last_updated_at` alone does **not establish a true data-freshness SLA**.

For example, a 20-minute difference between order creation and `last_updated_at` does not necessarily mean the system took 20 minutes to receive the status. The restaurant may simply have remained in that state for 20 minutes.

To measure actual freshness, the system should ideally capture separate timestamps such as:

```text
event_time   → when the real-world status changed
recorded_at  → when the application recorded the event
ingested_at  → when the data platform received the event
```

The actual freshness/ingestion delay could then be measured as:

```text
ingested_at - event_time
```

## Conclusion

Freshness is **use-case dependent** rather than a single property of the dataset.

The restaurant-status data has complete `last_updated_at` coverage and appears suitable for **weekly analytics**, assuming the reporting process has an appropriate cutoff. However, the available timestamps are **not sufficient to establish a near-real-time SLA for live customer ETA**.

For **restaurant accountability**, the data can provide useful evidence, but the 166 post-pickup updates and 6 chronology anomalies mean that individual status transitions should be validated before assigning responsibility for delays.

Therefore, the business should define **separate freshness SLAs for each use case** and instrument event time versus ingestion time if real-time freshness needs to be measured reliably.


# Challenge 6 — Build the validation gate


Summarize the investigation:

| Check | Status | Evidence | Action |
|---|---|---|---|
| Business grain |  |  |  |
| Timestamp chronology |  |  |  |
| KPI definition |  |  |  |
| Category semantics |  |  |  |
| Cross-source mapping |  |  |  |
| Freshness |  |  |  |

Use only:
**PASS / WARN / FAIL / UNKNOWN**

Finally answer:

> **Should leadership publish “Late Delivery Rate = 56%” today?**

What exactly must happen before the metric becomes publishable?

In [50]:
validation_report = {
    "business_grain": None,
    "timestamp_chronology": None,
    "kpi_definition": None,
    "category_semantics": None,
    "cross_source_mapping": None,
    "freshness": None,
    "publish_56_percent": None
}
print(validation_report)

{'business_grain': None, 'timestamp_chronology': None, 'kpi_definition': None, 'category_semantics': None, 'cross_source_mapping': None, 'freshness': None, 'publish_56_percent': None}


# Challenge 6 — Build the Validation Gate

## Validation Report

| Check                    | Status   | Evidence                                                                                                                                                                        | Action                                                                                                                               |
| ------------------------ | -------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------ |
| **Business grain**       | **WARN** | 1,603 rows but only 1,600 unique `order_id`s; 3 order IDs have duplicate records                                                                                                | Investigate and resolve/document duplicate order representations before using the KPI as an authoritative business metric            |
| **Timestamp chronology** | **WARN** | Missing timestamps exist; 4 `promised_eta` values occur before order creation, 5 pickup times occur after delivery, and 6 restaurant-status updates occur before order creation | Investigate chronology violations and define rules for invalid/missing timestamps; do not silently correct records                   |
| **KPI definition**       | **WARN** | Different valid definitions produce materially different rates: **56.39%** for delay > 0 minutes versus **23.34%** for delay > 10 minutes; no canonical KPI owner is documented | Operations, Support, Finance, and Data should agree on the definition, denominator, exclusions, and KPI owner                        |
| **Category semantics**   | **WARN** | Some differences are representation-only (`Delivered`/`delivered`, `HIGH`/`high`), while others may have different meanings (`handoff`/`handed_off`, `ETA issue`/`eta_changed`) | Normalize only proven representation differences; obtain business-owner confirmation before merging potentially different categories |
| **Cross-source mapping** | **PASS** | All tested relationships have **100% coverage**: restaurant, driver, ticket, and restaurant-status mappings                                                                     | No mapping remediation required for the relationships tested                                                                         |
| **Freshness**            | **WARN** | No missing `last_updated_at`; median creation → last status update is 20 minutes, but 6 chronology anomalies and 166 post-pickup updates exist                                  | Define use-case-specific freshness SLAs and instrument event time versus ingestion time for true freshness measurement               |

## Should Leadership Publish “Late Delivery Rate = 56%” Today?

**No — not as an unqualified, authoritative KPI.**

The investigation shows that approximately **56% can be reproduced under the “any delay > 0 minutes” definition**, but the metric is not yet fully validated as the company's canonical KPI.

Before publishing it, the following must happen:

1. **Confirm the business grain**

   * Resolve or document the 3 duplicate `order_id`s.
   * Confirm that one `order_id` represents exactly one business order.

2. **Resolve timestamp-quality issues**

   * Investigate invalid chronology and missing timestamps.
   * Define how orders with missing or invalid ETA/delivery timestamps are handled.
   * Do not silently impute or delete problematic records.

3. **Agree on the KPI definition**

   * Decide whether “late” means any delay greater than 0 minutes or a threshold such as more than 10 minutes.
   * Define the denominator and treatment of cancelled/refunded orders and missing timestamps.
   * Formally assign a KPI owner.

4. **Validate category semantics**

   * Normalize clear representation differences.
   * Confirm ambiguous mappings such as `handoff` vs `handed_off` and `ETA issue` vs `eta_changed`.

5. **Confirm freshness requirements**

   * Define the freshness SLA for the intended use case.
   * If the metric is intended for operational/live decisions, capture event time and ingestion time separately.

### Publishability Gate

The 56% metric becomes **publishable as an authoritative KPI only after the business definition, denominator, data-quality rules, and KPI ownership are formally agreed and the remaining material data-quality issues are either resolved or explicitly documented.**

Until then, the dashboard should show the metric with its **definition and eligible population**, rather than simply displaying:

> **Late Delivery Rate = 56%**


# Final Takeaway

The job was not to make the data look clean.

The job was to decide:

> **Is this data safe enough for this decision, and what remains unresolved?**

Across the investigation, I found that a metric can be numerically reproducible without being fully validated for decision-making.

The **56% Late Delivery Rate** can be reproduced under one definition, but its publishability depends on unresolved questions around business grain, timestamp quality, KPI definition, category semantics, and freshness.

The key FDE approach is therefore:

**Validate → quantify uncertainty → identify risks → involve the right business owner → define the decision boundary → then publish or act.**

A trustworthy data product is not one with zero warnings. It is one where the **known limitations are explicit, understood, and appropriate for the decision being made.**
